# Data Generation for Customer Retention and Loyalty Analysis

This notebook generates a synthetic dataset simulating customer transactions for a D2C fashion brand. The dataset includes customers, products, orders, and order items, and intentionally introduces small data quality issues to simulate real-world scenarios.

In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta


## Generate Products Table

In [2]:
categories = ["Oversized Tee", "Shirt", "Hoodie", "Jacket", "Pants"]

collection_types = ["Core", "Festive", "Limited Edition"]


In [3]:
def generate_price():
    price_type = random.choice(["low", "medium", "premium"])
    
    if price_type == "low":
        return random.randint(700, 899)
    elif price_type == "medium":
        return random.randint(900, 1499)
    else:
        return random.randint(1500, 1999)


In [4]:
products = []

for i in range(1, 61):
    product = {
        "product_id": f"P{i:03}",
        "category": random.choice(categories),
        "price": generate_price(),
        "collection_type": random.choice(collection_types)
    }
    products.append(product)

products_df = pd.DataFrame(products)
products_df.head()


,product_id,category,price,collection_type
0,P001,Jacket,983,Festive
1,P002,Oversized Tee,775,Festive
2,P003,Oversized Tee,1601,Festive
3,P004,Shirt,1913,Limited Edition
4,P005,Hoodie,1841,Festive


In [5]:
products_df.to_csv("products.csv", index=False)


## Generate Customers Table

In [6]:
store_cities = ["Kochi", "TVM", "Calicut", "Bangalore", "Chennai"]

online_cities = [
    "Kochi", "TVM", "Calicut", "Bangalore", "Chennai",
    "Kollam", "Thrissur", "Kottayam", "Mysore",
    "Coimbatore", "Hyderabad", "Mumbai"
]

acquisition_channels = ["Organic", "Instagram", "Influencer", "Referral"]


In [7]:
def random_date(start, end):
    return start + timedelta(days=random.randint(0, (end - start).days))


In [8]:
start_date = datetime(2024, 7, 1)
end_date = datetime(2025, 12, 31)



In [9]:
customers = []

for i in range(1, 10001):
    customer = {
        "customer_id": f"C{i:05}",
        "city": random.choice(online_cities),
        "acquisition_channel": random.choice(acquisition_channels),
        "first_purchase_date": random_date(start_date, end_date)
    }
    customers.append(customer)

customers_df = pd.DataFrame(customers)
customers_df.head()


,customer_id,city,acquisition_channel,first_purchase_date
0,C00001,Coimbatore,Influencer,2025-05-20
1,C00002,Kochi,Referral,2025-06-08
2,C00003,Kochi,Referral,2025-08-26
3,C00004,Hyderabad,Referral,2024-09-25
4,C00005,Kollam,Referral,2025-04-14


In [10]:
customers_df.to_csv("customers.csv", index=False)


In [11]:
# introduce some inconsistent city names
customers_df.loc[customers_df.sample(frac=0.03).index, "city"] = "Cochin"

# introduce some missing acquisition channels
customers_df.loc[customers_df.sample(frac=0.02).index, "acquisition_channel"] = None

customers_df.to_csv("customers.csv", index=False)


In [12]:
customers_df.info()
customers_df["city"].value_counts().head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   customer_id          10000 non-null  object        
 1   city                 10000 non-null  object        
 2   acquisition_channel  9800 non-null   object        
 3   first_purchase_date  10000 non-null  datetime64[ns]
dtypes: datetime64[ns](1), object(3)
memory usage: 312.6+ KB


city
Mumbai      868
Kochi       833
Kottayam    824
Calicut     813
TVM         812
Name: count, dtype: int64

## Generate Orders Table

In [13]:
payment_methods = ["Prepaid", "COD", "UPI", "Pay Later"]
campaign_types = ["Organic", "Influencer", "Festive", "Movie Promotion"]
order_channels = ["Online", "Store"]


In [14]:
def random_order_date():
    return random_date(start_date, end_date)


In [15]:
import numpy as np

# Realistic order frequency distribution
order_counts = np.random.choice(
    [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    size=10000,
    p=[0.35, 0.25, 0.15, 0.10, 0.07, 0.04, 0.02, 0.01, 0.005, 0.005]
)

# Seasonal multiplier - higher orders in Onam (Aug-Sep) and Christmas (Dec)
def get_seasonal_weight(date):
    month = date.month
    if month in [8, 9]:   # Onam
        return 2.5
    elif month == 12:      # Christmas
        return 2.0
    elif month in [10, 11]: # post-Onam dip
        return 0.8
    return 1.0

# Campaign weights - Organic dominates, others vary
campaign_weights = [0.40, 0.20, 0.25, 0.15]  # Organic, Influencer, Festive, Movie Promotion

# Channel weights - 65% Online, 35% Store
channel_weights = [0.65, 0.35]

orders = []
order_counter = 1

for idx, customer in customers_df.iterrows():
    num_orders = order_counts[idx]
    first_date = customer['first_purchase_date']
    
    for _ in range(num_orders):
        order_date = random_date(first_date, end_date)
        seasonal_w = get_seasonal_weight(order_date)
        
        # Seasonal boost: skip some orders in low seasons
        if random.random() > min(seasonal_w / 2.5, 1.0):
            if seasonal_w < 1.0:
                continue
        
        order = {
            "order_id": f"O{order_counter:06}",
            "customer_id": customer["customer_id"],
            "order_date": order_date,
            "order_channel": np.random.choice(["Online", "Store"], p=channel_weights),
            "payment_method": random.choice(payment_methods),
            "campaign_type": np.random.choice(campaign_types, p=campaign_weights),
            "discount_amount": random.randint(0, 300),
            "shipping_cost": random.randint(40, 120)
        }
        orders.append(order)
        order_counter += 1

orders_df = pd.DataFrame(orders)

In [16]:
orders_df.to_csv("orders.csv", index=False)


In [17]:
# missing campaign types in some rows
orders_df.loc[orders_df.sample(frac=0.03).index, "campaign_type"] = None

orders_df.to_csv("orders.csv", index=False)


In [18]:
orders_df.info()
orders_df["payment_method"].value_counts()
orders_df["campaign_type"].value_counts(dropna=False)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21042 entries, 0 to 21041
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   order_id         21042 non-null  object        
 1   customer_id      21042 non-null  object        
 2   order_date       21042 non-null  datetime64[ns]
 3   order_channel    21042 non-null  object        
 4   payment_method   21042 non-null  object        
 5   campaign_type    20411 non-null  object        
 6   discount_amount  21042 non-null  int64         
 7   shipping_cost    21042 non-null  int64         
dtypes: datetime64[ns](1), int64(2), object(5)
memory usage: 1.3+ MB


campaign_type
Organic            8247
Festive            5135
Influencer         4028
Movie Promotion    3001
None                631
Name: count, dtype: int64

In [19]:
orders_df.to_csv("orders.csv", index=False)


In [20]:
import os
os.listdir()


['.ipynb_checkpoints',
 '01_data_quality_and_cleaning.sql',
 '02_business_metrics.sql',
 '03_retention_rfm.sql',
 '04_campaign_omnichannel.sql',
 '05_time_analysis.sql',
 'Business Context & Problem.pdf',
 'customers.csv',
 'Customer_Retention_Dashboard.pbix',
 'data',
 'Data_Generation.ipynb',
 'orders.csv',
 'order_items.csv',
 'products.csv']

## Generate Order Items Table

In [21]:
import random

order_ids = orders_df["order_id"].tolist()
product_ids = products_df["product_id"].tolist()

order_items = []
item_counter = 1


In [22]:
for order_id in order_ids:
    num_items = random.randint(1, 3)
    
    for _ in range(num_items):
        product_id = random.choice(product_ids)
        quantity = random.randint(1, 2)

        # get price from products table
        price = products_df.loc[
            products_df["product_id"] == product_id, "price"
        ].values[0]

        order_items.append({
            "order_item_id": f"OI{item_counter:06d}",
            "order_id": order_id,
            "product_id": product_id,
            "quantity": quantity,
            "price": price
        })

        item_counter += 1


In [23]:
order_items_df = pd.DataFrame(order_items)
order_items_df.head()


,order_item_id,order_id,product_id,quantity,price
0,OI000001,O000001,P015,1,1811
1,OI000002,O000002,P019,2,799
2,OI000003,O000003,P057,2,892
3,OI000004,O000004,P046,2,722
4,OI000005,O000005,P020,1,1778


In [24]:
order_items_df.to_csv("order_items.csv", index=False)


In [25]:
len(order_items_df)


41979

In [26]:
import os
os.listdir()


['.ipynb_checkpoints',
 '01_data_quality_and_cleaning.sql',
 '02_business_metrics.sql',
 '03_retention_rfm.sql',
 '04_campaign_omnichannel.sql',
 '05_time_analysis.sql',
 'Business Context & Problem.pdf',
 'customers.csv',
 'Customer_Retention_Dashboard.pbix',
 'data',
 'Data_Generation.ipynb',
 'orders.csv',
 'order_items.csv',
 'products.csv']

In [27]:
customers_df.isnull().sum()
orders_df.isnull().sum()
order_items_df.isnull().sum()


order_item_id    0
order_id         0
product_id       0
quantity         0
price            0
dtype: int64

In [28]:
customers_df.isnull().sum()
orders_df.isnull().sum()


order_id             0
customer_id          0
order_date           0
order_channel        0
payment_method       0
campaign_type      631
discount_amount      0
shipping_cost        0
dtype: int64

In [29]:
customers_df.to_csv("customers.csv", index=False)
orders_df.to_csv("orders.csv", index=False)
products_df.to_csv("products.csv", index=False)
order_items_df.to_csv("order_items.csv", index=False)


In [30]:
import os
os.getcwd()


'C:\\Users\\hp\\Desktop\\Customer Retention and Loyalty Analysis'

In [31]:
import os

os.makedirs("data", exist_ok=True)


In [32]:
customers_df.to_csv("data/customers.csv", index=False)
products_df.to_csv("data/products.csv", index=False)
orders_df.to_csv("data/orders.csv", index=False)
order_items_df.to_csv("data/order_items.csv", index=False)


In [33]:
os.listdir("data")


['customers.csv', 'orders.csv', 'order_items.csv', 'products.csv']

In [34]:
order_freq = orders_df.groupby('customer_id')['order_id'].count()
print(order_freq.value_counts().sort_index())

order_id
1     3775
2     2352
3     1383
4      786
5      451
6      257
7      106
8       59
9       21
10       7
Name: count, dtype: int64


In [35]:
print("Total orders:", len(orders_df))
print("Avg orders per customer:", orders_df.groupby('customer_id').size().mean().round(2))
repeat = (orders_df.groupby('customer_id').size() > 1).mean() * 100
print("Repeat purchase rate:", round(repeat, 1), "%")

Total orders: 21042
Avg orders per customer: 2.29
Repeat purchase rate: 59.0 %


In [36]:
print("Total orders:", len(orders_df))
print("Avg orders per customer:", orders_df.groupby('customer_id').size().mean().round(2))
repeat = (orders_df.groupby('customer_id').size() > 1).mean() * 100
print("Repeat purchase rate:", round(repeat, 1), "%")

Total orders: 21042
Avg orders per customer: 2.29
Repeat purchase rate: 59.0 %


In [37]:
orders_per_customer = orders_df.groupby("customer_id").size()
orders_per_customer.value_counts().sort_index()

1     3775
2     2352
3     1383
4      786
5      451
6      257
7      106
8       59
9       21
10       7
Name: count, dtype: int64